# Calculate mortality at each grid point using central estimates only

$M(x, y) = POP(x, y) \; \times \; BMR_c \; \times \; AF(x, y)  $

In [1]:
import os
import xarray as xr
import numpy as np
from utils.mortality_utils import att_frac
from utils.mortality_utils import mortality
from utils.utils import get_scenario_config, create_global_country_map
import config
from utils.utils import require_dir
import pathlib

In [2]:
# === Path config ===
BMR_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "BMR")
POP_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "SSP_pop" / "SSP2")
MASK_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "BMR" / "masks" / "country")

In [3]:
# For file names
GBD_version = "GBD23"

In [5]:
# === Load data ===
bmr_file = f"{GBD_version}_BMR_Country_COPD_newlabels_1990-2009.nc"
bmr_path = os.path.join(BMR_DIR, bmr_file)
bmr_country = xr.open_dataarray(bmr_path).sel(quantile="mean")  # central estimate
BMR = create_global_country_map(bmr_country, MASK_DIR)

pop_file = "ssp2_coarse_grid_annual_2000-2100.nc"
pop_path = os.path.join(POP_DIR, pop_file)
population = xr.open_dataarray(pop_path)
pop = population.reindex_like(BMR, method="nearest", tolerance=1e-9)

The relative risk (RR) for a 10ppb increase in OSDMA8 is 1.074 [95% CI 1.012 - 1.137] and

$RR(x, y) = e^{\beta \, (OSDMA8(x, y) - TMREL)}$

In [6]:
# === Calculate beta for relative risk ===

# Relative risk for a 10ppb increase in OSDMA8, GBD 2023 - this has been the same since GBD21
RR_10ppb = 1.074  # central estimate
# equation is: RR = e^(beta*(x-TMREL)) where RR_10ppb = e^(10beta)
beta = np.log(RR_10ppb)/10

# TMREL from GBD 2021
TMREL = 32.4  # central estimate [95% Uniform CI 29.1 – 35.7]

In [7]:
# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
model = "CESM2"
scenario = "SSP245"

configs = get_scenario_config(model, scenario)
ensemble_members = configs["ensemble_members"]
years = configs["years"]

O3_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / model / "ozone" / "OSDMA8_BC")
SAVE_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / model / "mortality" / "ozone" / "gridpoint_mortality")

# === Main loop ===
for ens_num in ensemble_members:
    print(f"Processing {scenario}, Ensemble {ens_num:02d}")
    # {years.stop - 1} from OSDMA8 calculation
    dates = f"{years.start}-{years.stop - 1}"

    o3_file = f"OSDMA8_BC_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
    o3_path = os.path.join(O3_DIR, o3_file)
    o3 = xr.open_dataarray(o3_path)

    # Adjust indices to match (with small tolerance)
    # e.g., max 1e-7 km distance
    o3 = o3.reindex_like(BMR, method="nearest", tolerance=1e-9, fill_value=0)

    # Flag if any nans present (i.e. reindex was out of tolerance distance)
    assert not population.isnull().any()

    M = []

    for year in years:
        POP = pop.sel(year=year)
        # Calculate the attributable fraction
        AF = att_frac(o3.sel(year=year), TMREL, beta)
        # Calculate mortality at each grid point
        mortality_year = mortality(AF, BMR, POP)
        M.append(mortality_year)

    M_cleaned = [da.drop_vars("year", errors="ignore") for da in M]
    mortality_timeseries = xr.concat(
        M_cleaned,
        dim=(xr.DataArray(o3["year"].values,
                          dims="year", name="year")))

    description = ("Total COPD mortality due to surface ozone using "
                   "central estimates only - scripts "
                   "by A.F. Wells (2025)")
    mortality_timeseries.attrs["description"] = description
    mortality_timeseries.attrs["GBD version"] = GBD_version
    mortality_timeseries.attrs["ensemble_number"] = ens_num
    mortality_timeseries.attrs["scenario"] = scenario
    mortality_timeseries.attrs["model"] = model

    out_file = f"Mortality_{GBD_version}_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
    out_path = os.path.join(SAVE_DIR, out_file)

    print(f"Saving mortality timeseries to {out_path}")
    mortality_timeseries.to_netcdf(out_path)

print("All processing complete.")

Processing SSP245, Ensemble 01
Saving mortality timeseries to /glade/work/awells/workflow/CESM2/mortality/ozone/gridpoint_mortality/Mortality_GBD23_CESM2_SSP245_01_2020-2083.nc
Processing SSP245, Ensemble 02
Saving mortality timeseries to /glade/work/awells/workflow/CESM2/mortality/ozone/gridpoint_mortality/Mortality_GBD23_CESM2_SSP245_02_2020-2083.nc
Processing SSP245, Ensemble 03
Saving mortality timeseries to /glade/work/awells/workflow/CESM2/mortality/ozone/gridpoint_mortality/Mortality_GBD23_CESM2_SSP245_03_2020-2083.nc
All processing complete.
